In [ ]:

import torch
import os, random
import numpy as np
from transformers import set_seed

SEED = 159753


# 1) Python, NumPy, PyTorch
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 2) HF helper
set_seed(SEED)

# Criando base de dados

In [ ]:
import os
import glob

arch_folder_path = '/run/media/victor/pessoal/mestrado/dataset/arch'


arch_images_paths = glob.glob(os.path.join(arch_folder_path, '**', '*.*'), recursive=True)
arch_images_paths = [path for path in arch_images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]

In [ ]:
len(arch_images_paths)

In [ ]:
import os
import glob

leio_folder_path = '/run/media/victor/pessoal/mestrado/dataset/leiomioma'

leio_images_paths = glob.glob(os.path.join(leio_folder_path, '**', '*.*'), recursive=True)
leio_images_paths = [path for path in leio_images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]

In [ ]:
len(leio_images_paths)

In [ ]:
# path_mmu_folder_path = '/run/media/victor/pessoal/mestrado/dataset/pathMMU/images'

# path_mmu_images_paths = glob.glob(os.path.join(path_mmu_folder_path, '**', '*.*'), recursive=True)
# path_mmu_images_paths = [path for path in path_mmu_images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]


In [ ]:
# masson_folder_path = '/run/media/victor/pessoal/mestrado/dataset/Pathcap/masson_only'

# masson_images_paths = glob.glob(os.path.join(masson_folder_path, '**', '*.*'), recursive=True)
# masson_images_paths = [path for path in masson_images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]


In [ ]:
# path_mmu_images_paths

In [ ]:
# len(path_mmu_images_paths)

In [ ]:
# import os
# import glob

# plism_folder_path = r'C:\Users\victo\Documents\mestrado\dataset\Plism\PLISM_sm\PLISM_sm'

# plism_images_paths = glob.glob(os.path.join(plism_folder_path, '**', '*.*'), recursive=True)
# plism_images_paths = [path for path in plism_images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]

In [ ]:
# plism_images_paths

In [ ]:
temp_hist_paths = arch_images_paths + leio_images_paths

In [ ]:
len(temp_hist_paths)

In [ ]:
import os
from PIL import Image
import imagehash
from tqdm import tqdm

def get_duplicates(img_paths):
    
    hashes = {}

    for filepath in tqdm(img_paths):
        try:
            with Image.open(filepath) as img:
                hashes[filepath] = imagehash.phash(img)
        except Exception as e:
            print(f"Could not process {filepath}: {e}")

    similar_images = []
    file_list = list(hashes.keys())

    imgs_lists = []
    SIMILARITY_THRESHOLD = 5

    for i in tqdm(range(len(file_list))):
        is_in_a_group = any(file_list[i] in group for group in similar_images)
        
        if is_in_a_group:
            continue

        current_group = [file_list[i]]
        
        for j in range(i + 1, len(file_list)):
            distance = hashes[file_list[i]] - hashes[file_list[j]]
            
            if distance <= SIMILARITY_THRESHOLD:
                current_group.append(file_list[j])
                
        if len(current_group) > 1:
            similar_images.append(current_group)

        imgs_lists.append(current_group)

    return similar_images, imgs_lists


def get_unique_images(imgs_lists):
    unique_hist_paths= []

    for img_list in imgs_lists:
        unique_hist_paths.append(img_list[0])
    
    return unique_hist_paths


In [ ]:
similar_images, imgs_lists = get_duplicates(temp_hist_paths)

In [ ]:
unique_hist_paths = get_unique_images(imgs_lists)

In [ ]:
len(unique_hist_paths)

In [ ]:
import pandas as pd

pd.DataFrame(unique_hist_paths).to_csv('v2_unique_hist_paths.csv', index=False)

In [ ]:
from PIL import Image
import numpy as np

def is_majority_white(image_path, threshold=240, majority_percentage=0.5):
   
    try:

        img = Image.open(image_path).convert('L')

        img_array = np.array(img)
        

        total_pixels = img_array.size

        white_pixels = np.sum(img_array >= threshold)
        

        white_percentage = white_pixels / total_pixels
        

        return white_percentage > majority_percentage

    except FileNotFoundError:
        print(f"Error: The file '{image_path}' was not found.")
        return False
    except Exception as e:
        print(f"An error occurred: {e}")
        return False


In [ ]:
whites = []
hist_paths = []
for img in tqdm(unique_hist_paths):
    if is_majority_white(img, threshold=240, majority_percentage=0.5):
        whites.append(img)
    else:
        hist_paths.append(img)  


In [ ]:
len(whites)

In [ ]:
len(hist_paths)

In [ ]:
import numpy as np

train_image_paths = np.random.choice(hist_paths, size = int(len(hist_paths) * 0.8), replace = False)

In [ ]:
len(train_image_paths)

In [ ]:
non_train_arch_image_paths = [ path for path in hist_paths if path not in train_image_paths ]

In [ ]:
# train_image_paths = np.concatenate((train_image_paths, np.array(plism_images_paths)))

In [ ]:
len(non_train_arch_image_paths)

In [ ]:
len(train_image_paths)

In [ ]:
non_path_size = len(non_train_arch_image_paths)//4

## Ultrasound

In [ ]:
import os
import glob

ultra_sound_folder_path = '/run/media/victor/pessoal/mestrado/dataset/Fibroid Dataset'

ultra_sound_images_paths = glob.glob(os.path.join(ultra_sound_folder_path, '**', '*.jpg'), recursive=True)

In [ ]:
non_train_ultra_sound_images_paths = np.random.choice(ultra_sound_images_paths, size = min(non_path_size + 100, len(ultra_sound_images_paths)) , replace = False)

In [ ]:
len(non_train_ultra_sound_images_paths)

## Open Image

In [ ]:
import os
import glob

open_image_folder_path = '/run/media/victor/pessoal/mestrado/dataset/open_image'

open_image_images_paths = glob.glob(os.path.join(open_image_folder_path, '**', '*.jpg'), recursive=True)

In [ ]:
non_train_open_image_images_paths = np.random.choice(open_image_images_paths, size = non_path_size + 100, replace = False)

In [ ]:
len(non_train_open_image_images_paths)

## UMD

In [ ]:
import os
import glob

umd_folder_path = '/run/media/victor/pessoal/mestrado/dataset/UMD'

umd_images_paths = glob.glob(os.path.join(umd_folder_path, '**', '*.jpg'), recursive=True)

In [ ]:
non_train_umd_images_paths = np.random.choice(umd_images_paths, size = non_path_size + 100, replace = False)

In [ ]:
len(non_train_umd_images_paths)

## Cirurgias

In [ ]:
import os
import glob

surgery_folder_path = '/run/media/victor/pessoal/mestrado/dataset/LapGyn4_v1.2'

surgery_images_paths = glob.glob(os.path.join(surgery_folder_path, '**', '*.jpg'), recursive=True)

In [ ]:
non_train_surgery_images_paths = np.random.choice(surgery_images_paths, size = non_path_size + 100, replace = False)

In [ ]:
len(non_train_surgery_images_paths)

In [ ]:
non_train_df = pd.concat([
    
    pd.DataFrame({ "image_path" : non_train_arch_image_paths, "source" : "path"}),
    pd.DataFrame({ "image_path" : non_train_ultra_sound_images_paths, "source" : "ultra"}),
    pd.DataFrame({ "image_path" : non_train_umd_images_paths, "source" : "umd"}),
    pd.DataFrame({ "image_path" : non_train_surgery_images_paths, "source" : "surgery"}),
    pd.DataFrame({ "image_path" : non_train_open_image_images_paths, "source" : "open"}),


])

In [ ]:
len(non_train_df)

In [ ]:
similar_images_non_train, imgs_lists_non_train = get_duplicates(non_train_df['image_path'].values)

In [ ]:
unique_non_training = get_unique_images(imgs_lists_non_train)

In [ ]:
len(unique_non_training)

In [ ]:
non_train_unique_df = non_train_df[non_train_df["image_path"].isin(unique_non_training)]

In [ ]:
non_train_unique_df

In [ ]:
non_train_image_paths = (
    non_train_unique_df[non_train_unique_df["source"] == "path"]["image_path"].to_list()
    + non_train_unique_df[non_train_unique_df["source"] == "ultra"].sample(n=non_path_size)["image_path"].to_list()
    + non_train_unique_df[non_train_unique_df["source"] == "umd"].sample(n=non_path_size)["image_path"].to_list()
    + non_train_unique_df[non_train_unique_df["source"] == "surgery"].sample(n=non_path_size + 1)["image_path"].to_list()
     + non_train_unique_df[non_train_unique_df["source"] == "open"].sample(n=non_path_size + 1)["image_path"].to_list()
)

In [ ]:
len(non_train_image_paths)

In [ ]:
import pandas as pd

non_train_image_df = pd.DataFrame(non_train_image_paths, columns = ["image_path"])

In [ ]:
non_train_image_df

In [ ]:
for i, row in non_train_image_df.iterrows():
    if row["image_path"] in non_train_arch_image_paths:
        non_train_image_df.loc[i, "is_hist"] = 1
    else:
        non_train_image_df.loc[i, "is_hist"] = 0

In [ ]:
non_train_image_df['is_hist'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

X_test, X_val, y_test, y_val = train_test_split( non_train_image_df['image_path'].values, non_train_image_df['is_hist'].values, test_size=0.5, random_state=42, stratify=non_train_image_df['is_hist'].values)

In [ ]:
unique, counts = np.unique(y_test, return_counts=True)
dict(zip(unique, counts))

In [ ]:
unique, counts = np.unique(y_val, return_counts=True)
dict(zip(unique, counts))

In [ ]:
non_train_image_df.to_csv("v2_non_train_image.csv", index=False)

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),        
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize (ImageNet)
])

In [ ]:
# pre_transform = transforms.Compose([
#     transforms.Resize(256),
#     transforms.RandomCrop(224),                # Random crop
#     transforms.RandomVerticalFlip(),             # Random rotation up to 30 degrees
#     transforms.RandomHorizontalFlip(),         # Random horizontal flip
#     transforms.ColorJitter(
#         brightness=0.2, contrast=0.2,          # Random color jitter
#         saturation=0.2, hue=0.1
#     ),
#     transforms.ToTensor(),        
# ])

In [ ]:
# from torchvision import transforms

# train_transform = transforms.Compose([
#     transforms.ToTensor(),        
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize (ImageNet)
# ])

In [ ]:
# from PIL import Image
# from torchvision.utils import save_image
# from concurrent.futures import ThreadPoolExecutor
# from tqdm import tqdm
# import os

# train_aug_folder_path = r"C:\\Users\\victo\\Documents\\mestrado\\dataset\\augmentation\\"
# num_augmentation = 3

# def process_image(img):
#     image = Image.open(img).convert("RGB")
#     for i in range(num_augmentation):
#         augmented_image = pre_transform(image)
#         img_name = img.split("\\")[-1]
#         img_id = img_name.split(".")[0]
#         img_extension = img_name.split(".")[1]
#         save_path = train_aug_folder_path + img_id + fr"_aug_{i}." + img_extension
#         save_image(augmented_image, save_path)

# with ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
#     list(tqdm(executor.map(process_image, train_image_paths), total=len(train_image_paths)))

In [ ]:
# import os
# import glob

# train_aug_folder_path = r"C:\\Users\\victo\\Documents\\mestrado\\dataset\\augmentation\\"

# train_aug_images_paths = glob.glob(os.path.join(train_aug_folder_path, '**', '*.*'), recursive=True)
# train_aug_images_paths = [path for path in train_aug_images_paths if path.lower().endswith(('.jpg', '.jpeg', '.png'))]


In [ ]:
# len(train_aug_images_paths)

In [ ]:
# X_train = train_aug_images_paths

In [ ]:
X_train = train_image_paths

In [ ]:
import torchvision.models as models

model = models.convnext_tiny(weights='ConvNeXt_Tiny_Weights.IMAGENET1K_V1')


In [ ]:
model

In [ ]:
model.classifier=model.classifier[:-1]

In [ ]:
model

In [ ]:
import pandas as pd

# Create dataframes with image paths and labels
test_df = pd.DataFrame({'image_path': X_test, 'is_hist': y_test})
val_df = pd.DataFrame({'image_path': X_val, 'is_hist': y_val})
train_df = pd.DataFrame({'image_path': X_train})

# Save to CSV files
test_df.to_csv('v2_test_occ_set.csv', index=False)
val_df.to_csv('v2_val_occ_set.csv', index=False)
train_df.to_csv("v2_train_occ_set.csv", index=False)

In [ ]:

from dataloader_wrapper import  get_features


if __name__ == '__main__':
    model.to("cuda")
    train_outputs, test_outputs, val_outputs = get_features(X_train, X_test, X_val, model, transform, train_transform=transform)


In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
ss.fit(train_outputs)

X_train = ss.transform(train_outputs)
X_test = ss.transform(test_outputs)


In [ ]:
len(train_outputs)

In [ ]:
train_outputs[0].shape

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=512, whiten=True)
pca = pca.fit(X_train)
print('Explained variance percentage = %0.2f' % sum(pca.explained_variance_ratio_))
X_train = pca.transform(X_train)
X_test = pca.transform(X_test)


In [ ]:
from sklearn import svm

oc_svm_clf = svm.OneClassSVM(** {'gamma': 'scale', 'kernel': 'rbf', 'nu': 0.025})

In [ ]:
# {'gamma': 'auto', 'kernel': 'rbf', 'nu': 0.001,}

In [ ]:
# {'gamma': 'scale', 'kernel': 'rbf', 'nu': 0.05}

In [ ]:
oc_svm_clf.fit(X_train)

In [ ]:
oc_svm_preds = oc_svm_clf.predict(X_test)

In [ ]:
svm_if_results=pd.DataFrame({
  'oc_svm_preds': [0 if x == -1 else 1 for x in oc_svm_preds],
  'real' : y_test
})

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns

print(classification_report(svm_if_results['real'].values, svm_if_results['oc_svm_preds'].values))
sns.heatmap(confusion_matrix(svm_if_results['real'].values, svm_if_results['oc_svm_preds'].values),annot=True,fmt='2.0f')

In [ ]:
from sklearn.metrics import recall_score

recall_score(svm_if_results['real'].values, svm_if_results['oc_svm_preds'].values)

In [ ]:
from sklearn.metrics import precision_score

precision_score(svm_if_results['real'].values, svm_if_results['oc_svm_preds'].values)

In [ ]:
unique, counts = np.unique(y_test, return_counts=True)
dict(zip(unique, counts))

In [ ]:
# from sklearn.model_selection import ParameterGrid
# from joblib import Parallel, delayed 
# from sklearn.metrics import  precision_score

# param_grid = {
#     'kernel': [ 'rbf'],
#     'gamma': ['scale', 'auto', 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1],
#     'nu': [0.01, 0.025, 0.05, 0.1, 0.5, 0.9]
# }

# param_candidates = ParameterGrid(param_grid)
# print(f'{len(param_candidates)} candidates')

# def fit_model(params):
#     print(params)
#     model = svm.OneClassSVM(**params)
#     model.fit(X_train)
#     y_pred = model.predict(X_test)

#     y_pred =  [0 if y == -1 else 1 for y in y_pred]
#     score = precision_score(y_test, y_pred)
#     return [params, score]

# results = Parallel(n_jobs=-1, verbose=50)(delayed(fit_model)(params) for params in param_candidates)
# print(max(results, key=lambda x: x[1]))

In [ ]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([('scaler', ss), ('pca',pca), ('occ_model', oc_svm_clf)])

In [ ]:
oc_svm_preds_val = pipe.predict(val_outputs)

In [ ]:
svm_if_results_val=pd.DataFrame({
  'oc_svm_preds': [0 if x == -1 else 1 for x in oc_svm_preds_val],
  'real' : y_val
})

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns

print(classification_report(svm_if_results_val['real'].values, svm_if_results_val['oc_svm_preds'].values))
sns.heatmap(confusion_matrix(svm_if_results_val['real'].values, svm_if_results_val['oc_svm_preds'].values),annot=True,fmt='2.0f')

In [ ]:
import pickle


with open('/run/media/victor/pessoal/mestrado/codigo/model/V2_hist_occ.pkl','wb') as f:
    pickle.dump(pipe,f)